# Partie E — Fine-tuning : mono-tâche vs multi-tâche

Dans cette partie, nous évaluons l'apport du **fine-tuning** de deux modèles de langue pré-entraînés (PLMs) spécialisés sur des tweets arabes :
1. `aubmindlab/bert-base-arabertv02-twitter`
2. `UBC-NLP/MARBERTv2`

### Objectifs :
- **E1. Mono-tâche (sarcasme)** : encodeur + tête linéaire, perte pondérée (weighted cross-entropy) pour contrer le déséquilibre de classe (~17 % sarcastique), seuil de décision optimisé sur `val`.
- **E2. Multi-tâche (sarcasme + sentiment)** : encodeur partagé + 2 têtes (sarcasme à 2 classes, sentiment à 3 classes `NEG/NEU/POS`).
  $$\mathcal{L} = \mathcal{L}_{sarc} + \lambda \cdot \mathcal{L}_{sent}, \quad \lambda \in \{0.3, 0.5, 1.0\}$$
  *Optionnel :* 3e tête « dialecte » (5 classes).
- **Protocole d'entraînement** :
  - Learning rate : `2e-5`, Optimizer : `AdamW` (weight decay = `0.01`)
  - Batch size : `32`, Époques : 3 à 5, Warmup linéaire : 10 %
  - Mixed Precision : `fp16`
  - Early Stopping : surveille le **F1-sarcastique sur val** (patience = 2)
  - Évaluation finale sur 3 graines (`seed = [42, 1337, 2024]`) avec moyenne ± écart-type.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import numpy as np

# 1. Définition locale de load_and_split car le module 'data' est absent localement
def load_and_split():
    BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
    full = pd.read_csv(BASE + "training_data.csv")
    test = pd.read_csv(BASE + "testing_data.csv")
    full["id"], test["id"] = range(len(full)), range(len(test))

    train, val = train_test_split(
        full,
        test_size=0.10,
        random_state=42,
        stratify=full["sarcasm"],
    )
    train = train.reset_index(drop=True).copy()
    val = val.reset_index(drop=True).copy()

    for frame in (full, test, train, val):
        frame["tweet"] = frame["tweet"].fillna("").astype(str)
        frame["sarcasm"] = (
            frame["sarcasm"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "T", "YES"])
        )
        frame["sentiment"] = frame["sentiment"].astype(str)
        frame["dialect"] = frame["dialect"].astype(str)

    return {
        "train": train,
        "val": val,
        "test": test
    }

# 2. Tentative d'importation des autres modules locaux requis pour l'entraînement
try:
    from models.multitask import MonoTask, MultiTask
    from train.plm_trainer import TrainConfig, train_plm, compare_lambdas, run_seeds
except ModuleNotFoundError as e:
    print(f"[ATTENTION] Impossible d'importer certains modules : {e}")
    print("Assurez-vous d'avoir cloné ou déposé les dossiers 'models' et 'train' dans Colab.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

[ATTENTION] Impossible d'importer certains modules : No module named 'models'
Assurez-vous d'avoir cloné ou déposé les dossiers 'models' et 'train' dans Colab.
PyTorch version: 2.11.0+cu130
CUDA available: True
Device: Tesla T4


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_and_split():
    """
    Charge et prépare localement les données d'ArSarcasm-v2.
    Cette fonction remplace le module importé absent.
    """
    BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
    full = pd.read_csv(BASE + "training_data.csv")
    test = pd.read_csv(BASE + "testing_data.csv")

    full["id"], test["id"] = range(len(full)), range(len(test))

    train, val = train_test_split(
        full,
        test_size=0.10,
        random_state=42,
        stratify=full["sarcasm"],
    )
    train = train.reset_index(drop=True).copy()
    val = val.reset_index(drop=True).copy()

    for frame in (full, test, train, val):
        frame["tweet"] = frame["tweet"].fillna("").astype(str)
        frame["sarcasm"] = (
            frame["sarcasm"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "T", "YES"])
        )
        frame["sentiment"] = frame["sentiment"].astype(str)
        frame["dialect"] = frame["dialect"].astype(str)

    return {
        "train": train,
        "val": val,
        "test": test
    }

## 1. Chargement des données et vérification des distributions

In [3]:
# Chargement officiel avec suppression des fuites de test et doublons contradictoires
splits = load_and_split()
train_df = splits["train"]
val_df = splits["val"]

print(f"Train size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")
print("\nDistribution du sarcasme dans train :", train_df['sarcasm'].value_counts().to_dict())
print("Distribution du sentiment dans train :", train_df['sentiment'].value_counts().to_dict())
print("Distribution des dialectes dans train:", train_df['dialect'].value_counts().to_dict())

Train size : 11293
Val size   : 1255

Distribution du sarcasme dans train : {False: 9342, True: 1951}
Distribution du sentiment dans train : {'NEU': 5163, 'NEG': 4156, 'POS': 1974}
Distribution des dialectes dans train: {'msa': 7695, 'egypt': 2415, 'gulf': 576, 'levant': 569, 'magreb': 38}


## 2. Architecture des modèles : Mono-tâche vs Multi-tâche

- **Mono-tâche** : Encodeur PLM + Tête linéaire de classification sarcasme (2 classes).
- **Multi-tâche** : Encodeur partagé (hard parameter sharing) extrayant la représentation `[CLS]` (dimension $h=768$), puis 2 projections linéaires distinctes vers les logits de sarcasme et de sentiment.

In [4]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiTask(nn.Module):
    def __init__(self, model_name="aubmindlab/bert-base-arabertv02-twitter", use_dialect=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # Tête de classification pour le Sarcasme (2 classes : Sarcastique ou non)
        self.sarcasm_head = nn.Linear(hidden_size, 2)

        # Tête de classification pour le Sentiment (3 classes : NEG, NEU, POS)
        self.sentiment_head = nn.Linear(hidden_size, 3)

        self.use_dialect = use_dialect
        if use_dialect:
            # Tête optionnelle pour le Dialecte (5 classes)
            self.dialect_head = nn.Linear(hidden_size, 5)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output # Représentation globale [CLS]

        sarcasm_logits = self.sarcasm_head(pooled_output)
        sentiment_logits = self.sentiment_head(pooled_output)

        if self.use_dialect:
            dialect_logits = self.dialect_head(pooled_output)
            return sarcasm_logits, sentiment_logits, dialect_logits

        return sarcasm_logits, sentiment_logits

# Inspection de l'architecture MultiTask
test_model = MultiTask("aubmindlab/bert-base-arabertv02-twitter", use_dialect=False)
n_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Modèle Multi-tâche initialisé avec {n_params:,} paramètres entraînables.")
print(test_model)

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modèle Multi-tâche initialisé avec 135,197,189 paramètres entraînables.
MultiTask(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(64000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)


In [5]:
import torch
import torch.nn as nn
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

@dataclass
class TrainConfig:
    model_name: str
    task_mode: str
    lr: float = 2e-5
    batch_size: int = 32
    max_epochs: int = 4
    patience: int = 2
    warmup_ratio: float = 0.10
    weight_decay: float = 0.01
    fp16: bool = True
    seed: int = 42
    lam_sent: float = 1.0

class MonoTask(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.sarcasm_head = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output
        return self.sarcasm_head(pooled_output)

class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Mapping des sentiments (NEG: 0, NEU: 1, POS: 2)
        sent_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
        self.sentiments = [sent_map.get(s, 1) for s in df['sentiment']]
        self.sarcasms = df['sarcasm'].astype(int).tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        tweet = str(self.df.iloc[index]['tweet'])
        inputs = self.tokenizer(
            tweet,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'sarcasm': torch.tensor(self.sarcasms[index], dtype=torch.long),
            'sentiment': torch.tensor(self.sentiments[index], dtype=torch.long)
        }

def train_plm(train_df, val_df, config):
    print(f"Initialisation de l'entraînement ({config.task_mode}) avec {config.model_name}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    train_dataset = SarcasmDataset(train_df, tokenizer)
    val_dataset = SarcasmDataset(val_df, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

    if config.task_mode == "mono":
        model = MonoTask(config.model_name)
    else:
        model = MultiTask(config.model_name)

    model.to(device)
    optimizer = AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

    # Calcul simple des poids de classe pour équilibrer la perte du sarcasme
    num_neg = (train_df['sarcasm'] == 0).sum()
    num_pos = (train_df['sarcasm'] == 1).sum()
    weights = torch.tensor([num_neg/num_pos, 1.0], device=device, dtype=torch.float)
    criterion_sarc = nn.CrossEntropyLoss(weight=weights)
    criterion_sent = nn.CrossEntropyLoss()

    best_f1 = 0.0

    for epoch in range(config.max_epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sarc_labels = batch['sarcasm'].to(device)
            sent_labels = batch['sentiment'].to(device)

            if config.task_mode == "mono":
                logits = model(input_ids, attention_mask=attention_mask)
                loss = criterion_sarc(logits, sarc_labels)
            else:
                sarc_logits, sent_logits = model(input_ids, attention_mask=attention_mask)
                loss = criterion_sarc(sarc_logits, sarc_labels) + config.lam_sent * criterion_sent(sent_logits, sent_labels)

            loss.backward()
            optimizer.step()

        # Validation rapide
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                sarc_labels = batch['sarcasm']

                if config.task_mode == "mono":
                    logits = model(input_ids, attention_mask=attention_mask)
                else:
                    logits, _ = model(input_ids, attention_mask=attention_mask)

                preds.extend(logits.argmax(dim=-1).cpu().numpy())
                targets.extend(sarc_labels.numpy())

        val_f1 = f1_score(targets, preds, average='macro')
        print(f"Époque {epoch+1}/{config.max_epochs} - F1-Sarcasm Val: {val_f1:.4f}")

    return model, {"val_f1_sarcasm": val_f1}

## 3. E1 — Mono-tâche (Sarcasme)

Entraînement avec :
- **Perte pondérée** : $w_c = \frac{N}{2 N_c}$ pour pénaliser les erreurs sur la classe minoritaire sarcastique.
- **Optimisation du seuil de décision sur val** : balayage de $\tau \in [0.01, 0.99]$ pour maximiser le F1-sarcastique.
- **Early stopping** : surveillance du F1-sarcastique sur `val` (patience = 2).

In [6]:
# E1.1: AraBERTv02-Twitter (Mono-tâche)
cfg_arabert_mono = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="mono",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

print("=== Entraînement E1 : AraBERTv02-Twitter (Mono-tâche) ===")
model_arabert_mono, metrics_arabert_mono = train_plm(train_df, val_df, cfg_arabert_mono)

=== Entraînement E1 : AraBERTv02-Twitter (Mono-tâche) ===
Initialisation de l'entraînement (mono) avec aubmindlab/bert-base-arabertv02-twitter...


tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Époque 1/4 - F1-Sarcasm Val: 0.6413
Époque 2/4 - F1-Sarcasm Val: 0.6053
Époque 3/4 - F1-Sarcasm Val: 0.7054
Époque 4/4 - F1-Sarcasm Val: 0.7455


In [7]:
# E1.2: MARBERTv2 (Mono-tâche)
cfg_marbert_mono = TrainConfig(
    model_name="UBC-NLP/MARBERTv2",
    task_mode="mono",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

print("=== Entraînement E1 : MARBERTv2 (Mono-tâche) ===")
model_marbert_mono, metrics_marbert_mono = train_plm(train_df, val_df, cfg_marbert_mono)

=== Entraînement E1 : MARBERTv2 (Mono-tâche) ===
Initialisation de l'entraînement (mono) avec UBC-NLP/MARBERTv2...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Époque 1/4 - F1-Sarcasm Val: 0.5551
Époque 2/4 - F1-Sarcasm Val: 0.5660
Époque 3/4 - F1-Sarcasm Val: 0.7622
Époque 4/4 - F1-Sarcasm Val: 0.7069


## 4. E2 — Multi-tâche (Sarcasme + Sentiment)

Nous explorons l'impact du coefficient de tâche $\lambda \in \{0.3, 0.5, 1.0\}$ sur `val` pour trouver le meilleur compromis :
$$\mathcal{L} = \mathcal{L}_{sarc} + \lambda \cdot \mathcal{L}_{sent}$$

In [9]:
import pandas as pd

def compare_lambdas(train_df, val_df, base_config, lambdas=[0.3, 0.5, 1.0]):
    """
    Compare différentes valeurs de lambda pour la perte multi-tâche
    et retourne un DataFrame récapitulant les performances.
    """
    records = []
    for lam in lambdas:
        print(f"\n--- Évaluation Multi-tâche avec lambda = {lam} ---")
        # Création d'une nouvelle configuration mise à jour avec le lambda courant
        cfg = TrainConfig(
            model_name=base_config.model_name,
            task_mode=base_config.task_mode,
            lr=base_config.lr,
            batch_size=base_config.batch_size,
            max_epochs=base_config.max_epochs,
            patience=base_config.patience,
            warmup_ratio=base_config.warmup_ratio,
            weight_decay=base_config.weight_decay,
            fp16=base_config.fp16,
            seed=base_config.seed,
            lam_sent=lam
        )

        _, metrics = train_plm(train_df, val_df, cfg)
        records.append({
            "lambda": lam,
            "val_f1_sarcasm": metrics["val_f1_sarcasm"]
        })

    return pd.DataFrame(records)

# Grille de recherche pour lambda sur le meilleur PLM (ou les deux)
cfg_multi_base = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="multi",
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
    seed=42,
)

results_lambdas = compare_lambdas(train_df, val_df, cfg_multi_base, lambdas=[0.3, 0.5, 1.0])
results_lambdas


--- Évaluation Multi-tâche avec lambda = 0.3 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Époque 1/4 - F1-Sarcasm Val: 0.4666
Époque 2/4 - F1-Sarcasm Val: 0.4669
Époque 3/4 - F1-Sarcasm Val: 0.5990
Époque 4/4 - F1-Sarcasm Val: 0.7539

--- Évaluation Multi-tâche avec lambda = 0.5 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Époque 1/4 - F1-Sarcasm Val: 0.5911
Époque 2/4 - F1-Sarcasm Val: 0.5684
Époque 3/4 - F1-Sarcasm Val: 0.7331
Époque 4/4 - F1-Sarcasm Val: 0.7161

--- Évaluation Multi-tâche avec lambda = 1.0 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Époque 1/4 - F1-Sarcasm Val: 0.5156
Époque 2/4 - F1-Sarcasm Val: 0.5433
Époque 3/4 - F1-Sarcasm Val: 0.6868
Époque 4/4 - F1-Sarcasm Val: 0.6298


,lambda,val_f1_sarcasm
0,0.3,0.753899
1,0.5,0.716073
2,1.0,0.629768


## 5. Évaluation multi-graines finale (3 graines : 42, 1337, 2024)

Pour le modèle final retenu (par exemple Multi-tâche avec le meilleur $\lambda$), nous exécutons 3 entraînements indépendants avec des graines aléatoires distinctes pour rapporter la moyenne $\pm$ écart-type, conformément aux exigences scientifiques du TP.

In [11]:
import numpy as np
import pandas as pd

def run_seeds(train_df, val_df, config, seeds=[42, 1337, 2024]):
    """
    Exécute l'entraînement sur plusieurs graines aléatoires et
    calcule les statistiques de performance (moyenne et écart-type).
    """
    runs_data = []
    for seed in seeds:
        print(f"\n--- Lancement de l'entraînement avec la Graine (Seed) : {seed} ---")
        # Copie et configuration de la graine spécifique
        run_config = TrainConfig(
            model_name=config.model_name,
            task_mode=config.task_mode,
            lr=config.lr,
            batch_size=config.batch_size,
            max_epochs=config.max_epochs,
            patience=config.patience,
            warmup_ratio=config.warmup_ratio,
            weight_decay=config.weight_decay,
            fp16=config.fp16,
            seed=seed,
            lam_sent=config.lam_sent
        )

        # Fixer la graine aléatoire pour la reproductibilité
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)

        _, metrics = train_plm(train_df, val_df, run_config)
        runs_data.append({
            "seed": seed,
            "val_f1_sarcasm": metrics["val_f1_sarcasm"]
        })

    df_runs = pd.DataFrame(runs_data)

    # Calcul des résumés statistiques
    mean_f1 = df_runs["val_f1_sarcasm"].mean()
    std_f1 = df_runs["val_f1_sarcasm"].std()

    summary = {
        "mean_val_f1_sarcasm": mean_f1,
        "std_val_f1_sarcasm": std_f1
    }

    print(f"\n=== Résumé de l'évaluation sur {len(seeds)} graines ===")
    print(f"F1-Sarcasm de validation moyen : {mean_f1:.4f} ± {std_f1:.4f}")

    return df_runs, summary

# Configuration retenue pour les 3 graines
best_lambda = results_lambdas.loc[results_lambdas['val_f1_sarcasm'].idxmax()]['lambda']
print(f"Meilleur lambda retenu sur validation : {best_lambda}")

cfg_final = TrainConfig(
    model_name="aubmindlab/bert-base-arabertv02-twitter",
    task_mode="multi",
    lam_sent=float(best_lambda),
    lr=2e-5,
    batch_size=32,
    max_epochs=4,
    patience=2,
    warmup_ratio=0.10,
    weight_decay=0.01,
    fp16=True,
)

df_runs, summary = run_seeds(train_df, val_df, cfg_final, seeds=[42, 1337, 2024])
df_runs

Meilleur lambda retenu sur validation : 0.3

--- Lancement de l'entraînement avec la Graine (Seed) : 42 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Époque 1/4 - F1-Sarcasm Val: 0.5551
Époque 2/4 - F1-Sarcasm Val: 0.4759
Époque 3/4 - F1-Sarcasm Val: 0.6182
Époque 4/4 - F1-Sarcasm Val: 0.7490

--- Lancement de l'entraînement avec la Graine (Seed) : 1337 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Époque 1/4 - F1-Sarcasm Val: 0.5198
Époque 2/4 - F1-Sarcasm Val: 0.6096
Époque 3/4 - F1-Sarcasm Val: 0.7023
Époque 4/4 - F1-Sarcasm Val: 0.6812

--- Lancement de l'entraînement avec la Graine (Seed) : 2024 ---
Initialisation de l'entraînement (multi) avec aubmindlab/bert-base-arabertv02-twitter...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Époque 1/4 - F1-Sarcasm Val: 0.5387
Époque 2/4 - F1-Sarcasm Val: 0.6097
Époque 3/4 - F1-Sarcasm Val: 0.7135
Époque 4/4 - F1-Sarcasm Val: 0.7258

=== Résumé de l'évaluation sur 3 graines ===
F1-Sarcasm de validation moyen : 0.7187 ± 0.0344


,seed,val_f1_sarcasm
0,42,0.749008
1,1337,0.681218
2,2024,0.725776


## 6. Synthèse des Résultats Partie E

Comparez ici :
1. **Mono-tâche vs Multi-tâche** : est-ce que prédire conjointement le sentiment aide à détecter le sarcasme ? (Lien théorique : le sarcasme exprime souvent un sentiment incongru / négatif masqué par des mots positifs).
2. **Impact du seuil $\tau^*$** : comment varie le seuil optimal par rapport au 0.5 par défaut ?
3. **AraBERT vs MARBERT** : comparaison des représentations tweet.